In [11]:
import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.colors as mcolors
import igl
from queue import PriorityQueue
# import matplotlib as mpl
# from matplotlib.colors import LinearSegmentedColormap, ListedColormap
# from distinctipy import distinctipy
import meshplot as mp
# import timeit
import openmesh as om
import trimesh
import os
import json
import sys



%load_ext autoreload
%autoreload 2
from utils import *
from primitive_fitting import *


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
# obj_path = "example_data/example_CAD/2.obj"
# patch_path = 'example_results/vsa/2_proxy_cluster.txt'
# region_path = 'example_results/vsa/2_region_cluster.txt'
# merge_path = 'example_results/vsa/2_merged_proxy.txt'
# raw_merge_path = 'example_results/vsa/2_raw_merged_proxy.txt'

obj_path = "example_data/ABC/00000006.obj"
patch_path = 'example_results/vsa/00000006_proxy_cluster.txt'
raw_patch_path = 'example_results/vsa/00000006_directly.txt'

patch2f = []
f = open(patch_path, 'r')
lines = f.readlines()
for line in lines:
    patch2f.append(np.array(line.split(), dtype=int))
raw_patch2f = []
f = open(raw_patch_path, 'r')
lines = f.readlines()
for line in lines:
    raw_patch2f.append(np.array(line.split(), dtype=int))
# patch2f, region_cluster = read_regioned_cluster(patch_path, region_path)
# patch2f, merge_cluster = read_regioned_cluster(patch_path, merge_path)
# patch2f, raw_merge_cluster = read_regioned_cluster(patch_path, raw_merge_path)
v, f = igl.read_triangle_mesh(obj_path)

f_area = igl.doublearea(v, f) / 2.0
f_center = igl.barycenter(v, f)
f_normals = igl.per_face_normals(v, f, np.array([0.0, 0.0, 1.0], dtype=np.float64))

mesh_surface_area = np.sum(f_area)

def patch_fitting_rate(patch, v, f):
    fitting = PrimitiveFitting(v, f, f_normals, f_center, f_area)
    planar_rate, _ = fitting.fit_planar(patch)
    cylinder_rate, _ = fitting.fit_cylinder(patch)
    sphere_rate, _ = fitting.fit_sphere(patch)
    # cone_rate, _ = fitting.fit_cone(range(len(f)))
    print(f"fitting rates: planar {planar_rate:.4f}, cylinder {cylinder_rate:.4f}, sphere {sphere_rate:.4f})")

    total_area = 0.0
    for f_id in patch:
        total_area += f_area[f_id]
    print(f"patch area: {total_area:.4f}")

    score = max(planar_rate, cylinder_rate, sphere_rate) * total_area
    return score

def cluster_fitting_score(patch2f, v, f):
    score = 0.0
    for patch in patch2f:
        score += patch_fitting_rate(patch, v, f)
    return score
print(f"patch2f score: {cluster_fitting_score(patch2f, v, f) / mesh_surface_area:.4f}")
print(f"raw_patch2f score: {cluster_fitting_score(raw_patch2f, v, f) / mesh_surface_area:.4f}")

# merged_patch2f = []
# for c in merge_cluster:
#     merged_patch = []
#     for patch in c:
#         merged_patch.extend(patch2f[patch])
#     merged_patch2f.append(merged_patch)
# print(f"merge_cluster score: {cluster_fitting_score(merged_patch2f, v, f) / mesh_surface_area:.4f}")

# raw_merged_patch2f = []
# for c in raw_merge_cluster:
#     raw_merged_patch = []
#     for patch in c:
#         raw_merged_patch.extend(patch2f[patch])
#     raw_merged_patch2f.append(raw_merged_patch)
# print(f"merge_cluster score: {cluster_fitting_score(raw_merged_patch2f, v, f) / mesh_surface_area:.4f}")


fitting rates: planar 1.0000, cylinder 1.0000, sphere 0.0000)
patch area: 4.0171
fitting rates: planar 1.0000, cylinder 1.0000, sphere 0.4159)
patch area: 4.0172
fitting rates: planar 1.0000, cylinder 1.0000, sphere 0.4014)
patch area: 4.0172
fitting rates: planar 0.8387, cylinder 1.0000, sphere 0.9935)
patch area: 0.4317
fitting rates: planar 0.8165, cylinder 1.0000, sphere 1.0000)
patch area: 0.4317
fitting rates: planar 0.8387, cylinder 1.0000, sphere 0.9806)
patch area: 0.4316
fitting rates: planar 0.8165, cylinder 1.0000, sphere 0.9747)
patch area: 0.4317
fitting rates: planar 1.0000, cylinder 1.0000, sphere 0.2217)
patch area: 12.8477
fitting rates: planar 1.0000, cylinder 1.0000, sphere 0.1678)
patch area: 14.6793
fitting rates: planar 1.0000, cylinder 1.0000, sphere 0.4014)
patch area: 4.0172
fitting rates: planar 0.0672, cylinder 1.0000, sphere 0.2279)
patch area: 12.5399
fitting rates: planar 1.0000, cylinder 1.0000, sphere 0.0000)
patch area: 4.0171
fitting rates: planar 1.0

d:\Desktop\MeshSegment\fitting_geometric_primitives\geometry_primitive.py:138: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  best_params = np.array([
